1.환경 셋업, 데이터 로드 완료

In [1]:
%pip install sentence_transformers langchain langchain-openai langchain_community openai tiktoken python-dotenv graphdatascience altair neo4j_tools
%pip install "vegafusion[embed]" 

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: sentence_transformers in c:\users\yujin\anaconda3\lib\site-packages (5.1.0)

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from graphdatascience import GraphDataScience
from neo4j_tools import gds_db_load, gds_utils
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores.neo4j_vector import Neo4jVector
from langchain.graphs import Neo4jGraph
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain.schema import StrOutputParser
from langchain.schema.runnable import RunnableLambda

# pandas 출력 옵션 설정 (표시 행 수, 너비 등)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_colwidth', 500)
pd.set_option('display.width', 0)

In [134]:
# Neo4j 설정값
NEO4J_URI = "bolt://127.0.0.1:7687" 
NEO4J_PASSWORD = 'love4773'
NEO4J_USERNAME = 'neo4j'
AURA_DS = False

In [4]:
# OpenAI API 키를 안전하게 입력받음
from dotenv import load_dotenv
import getpass
import os

LLM = 'gpt-5'
os.environ['OPENAI_API_KEY'] = getpass.getpass()

 ········


In [5]:
from graphdatascience import GraphDataScience

# gds 객체 생성
gds = GraphDataScience(
    "bolt://127.0.0.1:7687",
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    aura_ds=AURA_DS)
gds.set_database("neo4j")

In [117]:
# 1) 데이터 불러오기 + Neo4j 로딩 전체 파이프라인
import pandas as pd
from neo4j_tools import gds_db_load  # DataFrame을 Neo4j로 넣는 헬퍼
# gds는 미리 만든 GraphDataScience 인스턴스(Neo4j 연결 객체)라고 가정

# 2) 데이터 읽기
import pandas as pd
import numpy as np

ar = pd.read_csv('Desktop/H&M/articles.csv')
cu = pd.read_csv('Desktop/H&M/customers.csv')
tr = pd.read_csv('Desktop/H&M/transactions_train.csv')
ar.head(2)

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.


In [7]:
cu.head(2)

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,NaN,NaN,ACTIVE,NONE,49.0,52043ee2162cf5aa7ee79974281641c6f11a68d276429a91f8ca0d4b6efa8100
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,NaN,NaN,ACTIVE,NONE,25.0,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93f4c830291c32bc3057


In [8]:
tr.head(2)

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,541518023,0.030492,2


In [9]:
ar.columns

Index(['article_id', 'product_code', 'prod_name', 'product_type_no',
       'product_type_name', 'product_group_name', 'graphical_appearance_no',
       'graphical_appearance_name', 'colour_group_code', 'colour_group_name',
       'perceived_colour_value_id', 'perceived_colour_value_name',
       'perceived_colour_master_id', 'perceived_colour_master_name',
       'department_no', 'department_name', 'index_code', 'index_name',
       'index_group_no', 'index_group_name', 'section_no', 'section_name',
       'garment_group_no', 'garment_group_name', 'detail_desc'],
      dtype='object')

In [135]:
ar['product_type_name'].unique()

array(['Vest top', 'Bra', 'Underwear Tights', 'Socks', 'Leggings/Tights',
       'Sweater', 'Top', 'Trousers', 'Hair clip', 'Umbrella',
       'Pyjama jumpsuit/playsuit', 'Bodysuit', 'Hair string', 'Unknown',
       'Hoodie', 'Sleep Bag', 'Hair/alice band', 'Belt', 'Boots',
       'Bikini top', 'Swimwear bottom', 'Underwear bottom', 'Swimsuit',
       'Skirt', 'T-shirt', 'Dress', 'Hat/beanie', 'Kids Underwear top',
       'Shorts', 'Shirt', 'Cap/peaked', 'Pyjama set', 'Sneakers',
       'Sunglasses', 'Cardigan', 'Gloves', 'Earring', 'Bag', 'Blazer',
       'Other shoe', 'Jumpsuit/Playsuit', 'Sandals', 'Jacket', 'Costumes',
       'Robe', 'Scarf', 'Coat', 'Other accessories', 'Polo shirt',
       'Slippers', 'Night gown', 'Alice band', 'Straw hat', 'Hat/brim',
       'Tailored Waistcoat', 'Necklace', 'Ballerinas', 'Tie',
       'Pyjama bottom', 'Felt hat', 'Bracelet', 'Blouse',
       'Outdoor overall', 'Watch', 'Underwear body', 'Beanie', 'Giftbox',
       'Sleeping sack', 'Dungarees',

In [11]:
cu.columns

Index(['customer_id', 'FN', 'Active', 'club_member_status',
       'fashion_news_frequency', 'age', 'postal_code'],
      dtype='object')

In [12]:
tr.columns

Index(['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id'], dtype='object')

In [118]:
# 최근 1일 구매된 상품 기준 필터링
from datetime import datetime, timedelta

# 날짜 컬럼을 datetime으로 변환
tr['t_dat'] = pd.to_datetime(tr['t_dat'])

# 가장 최근 날짜 구하기
latest_day = tr['t_dat'].max().normalize()   # 시분초 제거, 날짜만

#test 데이터

In [119]:
# 최근 1일 거래만 필터링
tr_test = tr.loc[tr['t_dat'].dt.normalize() == latest_day].copy()
tr_test.shape

(32866, 5)

In [15]:
# test에 등장한 고객 ID "리스트" 만들기
custid_list = tr_test['customer_id'].unique().tolist()
len(custid_list)

10528

#train데이터

In [120]:
# 최근 7일 범위 (latest_day 제외)
start_day = latest_day - timedelta(days=7)

# 최근 7일 거래만 필터링 (train)
tr = tr[(tr['t_dat'].dt.normalize() >= start_day) &
              (tr['t_dat'].dt.normalize() < latest_day)].copy()
tr = tr[tr['customer_id'].isin(custid_list)].copy()
tr.shape

(9529, 5)

In [17]:
# 최근 7일 동안 구매된 article_id 추출
recent_article_ids = tr[(tr['t_dat'].dt.normalize() >= start_day) &
                        (tr['t_dat'].dt.normalize() <= latest_day)]['article_id'].unique()

# ar에서 최근 7일 구매 상품만 필터링
ar_recent = ar[ar['article_id'].isin(recent_article_ids)]
ar_recent = ar_recent[ar_recent['detail_desc'].notnull()]

# 최종 결과
ar = ar_recent
ar.shape

(4245, 25)

In [18]:
# cu 중에서 test 고객만 필터링
cu_recent = cu[cu['customer_id'].isin(custid_list)]
cu = cu_recent

cu.shape

(10528, 7)

In [19]:
# 3) 고유 제약(UNIQUE CONSTRAINT) 생성
#    - 각 라벨별 "식별자" 속성에 유일성 보장
#    - 중복 삽입 방지, merge 시 키로 사용
gds.run_cypher('CREATE CONSTRAINT unique_department_no IF NOT EXISTS FOR (n:Department) REQUIRE n.departmentNo IS UNIQUE')
gds.run_cypher('CREATE CONSTRAINT unique_product_code  IF NOT EXISTS FOR (n:Product)   REQUIRE n.product_code  IS UNIQUE')
gds.run_cypher('CREATE CONSTRAINT unique_article_id    IF NOT EXISTS FOR (n:Article)   REQUIRE n.articleId    IS UNIQUE')
gds.run_cypher('CREATE CONSTRAINT unique_customer_id   IF NOT EXISTS FOR (n:Customer)  REQUIRE n.customerId   IS UNIQUE')

# 4) 노드 로드
gds_db_load.load_nodes(gds, ar[['department_no', 'department_name']], 'department_no', 'Department')  # Department
gds_db_load.load_nodes(gds, ar.drop(columns=['product_code', 'department_no']), 'article_id', 'Article')  # Article
gds_db_load.load_nodes(gds, ar[['product_code', 'prod_name', 'product_type_name', 'product_group_name', 'garment_group_name', 'detail_desc']], 'product_code', 'Product')  # Product
gds_db_load.load_nodes(gds, cu, 'customer_id', 'Customer')  # Customer

# 5) 관계 로드
gds_db_load.load_rels(
    gds,
    ar[['article_id', 'department_no']],
    source_target_labels=('Article', 'Department'),
    source_node_key='article_id',
    target_node_key='department_no',
    rel_type='FROM_DEPARTMENT'
)

gds_db_load.load_rels(
    gds,
    ar[['article_id', 'product_code']],
    source_target_labels=('Article', 'Product'),
    source_node_key='article_id',
    target_node_key='product_code', #디자인/스타일은 같지만 색상·사이즈는 다를 수 있음
    rel_type='VARIANT_OF'
)

# txId 생성
tr['txId'] = tr.index.astype(str)

gds_db_load.load_rels(
    gds,
    tr[['customer_id', 'article_id', 't_dat', 'txId']],
    source_target_labels=('Customer', 'Article'),
    source_node_key='customer_id',
    target_node_key='article_id',
    rel_key='txId',
    rel_type='PURCHASED'
)

# 6) 날짜 컬럼 타입 변환
gds.run_cypher('''
MATCH (:Customer)-[r:PURCHASED]->()
SET r.tDat = date(r.tDat)
''')

# 7) Product 설명 텍스트 생성
gds.run_cypher("""
MATCH (p:Product)
SET p.text = '##Product\n' +
             'Name: '         + coalesce(p.prodName, '')          + '\n' +
             'Type: '         + coalesce(p.productTypeName, '')    + '\n' +
             'Group: '        + coalesce(p.productGroupName, '')   + '\n' +
             'Garment Type: ' + coalesce(p.garmentGroupName, '')   + '\n' +
             'Description: '  + coalesce(p.detailDesc, '')
RETURN count(p) AS propertySetCount
""")

======  loading Department nodes  ======
staging 4,245 records

Using This Cypher Query:
```
UNWIND $recs AS rec
MERGE(n:Department {department_no: rec.department_no})
SET n.department_name = rec.department_name
RETURN count(n) AS nodeLoadedCount
```

Loaded 4,245 of 4,245 nodes
======  loading Article nodes  ======
staging 4,245 records

Using This Cypher Query:
```
UNWIND $recs AS rec
MERGE(n:Article {article_id: rec.article_id})
SET n.prod_name = rec.prod_name, n.product_type_no = rec.product_type_no, n.product_type_name = rec.product_type_name, n.product_group_name = rec.product_group_name, n.graphical_appearance_no = rec.graphical_appearance_no, n.graphical_appearance_name = rec.graphical_appearance_name, n.colour_group_code = rec.colour_group_code, n.colour_group_name = rec.colour_group_name, n.perceived_colour_value_id = rec.perceived_colour_value_id, n.perceived_colour_value_name = rec.perceived_colour_value_name, n.perceived_colour_master_id = rec.perceived_colour_master_id, n

,propertySetCount
0,2771


2. 오픈 API 호출하여 text 임베딩하기

In [20]:
from langchain_openai import OpenAIEmbeddings

# 임베딩 모델 초기화
embedding_model = OpenAIEmbeddings()
# 임베딩 벡터 차원
embedding_dimension = 1536

In [21]:
# 판다스 출력 옵션 설정
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_colwidth', 500)
pd.set_option('display.width', 0)

# ar에서 제품 정보만 추출 (productCode 기준으로 중복 제거)
product_emb_df = ar[['product_code', 'prod_name', 'product_type_name',
                     'product_group_name', 'garment_group_name', 'detail_desc']]

# detail_desc가 비어있지 않은 행만 남김
product_emb_df = product_emb_df[product_emb_df.detail_desc.notnull()]

# 개별 행(row)을 받아서 하나의 텍스트 문서 형태로 변환하는 함수 정의
def create_doc(row):
    return f'''
##Product
Name: {row.prod_name}
Type: {row.product_type_name}
Group: {row.product_group_name}
Garment Type: {row.garment_group_name}
Description: {row.detail_desc}
'''

# 'text' 컬럼 생성
product_emb_df['text'] = product_emb_df.apply(create_doc, axis=1)

# 원래 컬럼 제거 후 product_code와 text만 남김
product_emb_df = product_emb_df.drop(columns=['prod_name', 'product_type_name',
                                              'product_group_name', 'garment_group_name', 'detail_desc'])

# 최종 결과 확인
product_emb_df

,product_code,text
6,111565,"\n##Product\nName: 20 den 1p Stockings\nType: Underwear Tights\nGroup: Socks & Tights\nGarment Type: Socks and Tights\nDescription: Semi shiny nylon stockings with a wide, reinforced trim at the top. Use with a suspender belt. 20 denier.\n"
8,111586,\n##Product\nName: Shape Up 30 den 1p Tights\nType: Leggings/Tights\nGroup: Garment Lower body\nGarment Type: Socks and Tights\nDescription: Tights with built-in support to lift the bottom. Black in 30 denier and light amber in 15 denier.\n
27,123173,\n##Product\nName: Control Top 50 den 1p Tights\nType: Leggings/Tights\nGroup: Garment Lower body\nGarment Type: Socks and Tights\nDescription: 50 denier tights with reinforcement at the top for a shaping effect on the tummy and thighs.\n
35,129085,\n##Product\nName: Pirate Leggings (1)\nType: Leggings/Tights\nGroup: Garment Lower body\nGarment Type: Jersey Basic\nDescription: 3/4-length leggings in stretch jersey with an elasticated waist.\n
58,153115,"\n##Product\nName: OP Strapless^\nType: Bra\nGroup: Underwear\nGarment Type: Under-, Nightwear\nDescription: Strapless bra in microfibre with underwired, padded cups that lift and shape the bust. Silicone trim at the top and a hook-and-eye fastening at the back. Detachable, adjustable shoulder straps and side support.\n"
...,...,...
105528,949198,"\n##Product\nName: Saturday jogger\nType: Trousers\nGroup: Garment Lower body\nGarment Type: Jersey Basic\nDescription: Joggers in sweatshirt fabric made from a cotton blend with an elasticated, drawstring waist, discreet side pockets and straight legs. Soft brushed inside. The polyester content of the joggers is recycled.\n"
105530,949551,"\n##Product\nName: Virgo sweater fast buy\nType: Sweater\nGroup: Garment Upper body\nGarment Type: Jersey Fancy\nDescription: Short, boxy-style top in sweatshirt fabric made from a cotton blend with dropped shoulders and long sleeves with ribbing at the cuffs. Smocking at the hem. Soft brushed inside.\n"
105536,952938,\n##Product\nName: Elton top\nType: Top\nGroup: Garment Upper body\nGarment Type: Jersey Fancy\nDescription: Fitted top in jersey with a round neckline and extra-long sleeves. Additional draped layer at the front.\n
105538,953763,\n##Product\nName: SPORT Malaga tank\nType: Vest top\nGroup: Garment Upper body\nGarment Type: Jersey Fancy\nDescription: Loose-fitting sports vest top in ribbed fast-drying functional fabric made from recycled polyester with a racer back and rounded hem.\n


In [22]:
%%time 

count = 0
embeddings = []

# product_emb_df.text 컬럼에 들어있는 문서들을 500개씩 잘라서(chunking) 처리
# - 한번에 너무 많은 텍스트를 임베딩하면 API 한도/성능 문제 발생
# - chunk 크기(n=500)는 정확도와 직접 관계는 없고 효율/안정성을 위한 것
for docs in gds_db_load.chunks(product_emb_df.text, n=500):
    count += len(docs)  # 지금까지 처리한 문서 개수 누적
    print(f'Embedded {count} of {product_emb_df.shape[0]}')  
    # 현재까지 몇 개 임베딩했는지 출력
    
    # OpenAI 임베딩 모델을 호출해서 500개 단위로 임베딩 벡터 생성
    embeddings.extend(embedding_model.embed_documents(docs))

# 임베딩 결과(숫자 벡터 리스트)를 DataFrame에 새로운 컬럼으로 추가
product_emb_df['textEmbedding'] = embeddings

# 최종적으로 product_emb_df를 반환 (임베딩된 텍스트 포함된 DataFrame)
product_emb_df

Embedded 500 of 4245
Embedded 1000 of 4245
Embedded 1500 of 4245
Embedded 2000 of 4245
Embedded 2500 of 4245
Embedded 3000 of 4245
Embedded 3500 of 4245
Embedded 4000 of 4245
Embedded 4245 of 4245
CPU times: total: 1.86 s
Wall time: 24.9 s


,product_code,text,textEmbedding
6,111565,"\n##Product\nName: 20 den 1p Stockings\nType: Underwear Tights\nGroup: Socks & Tights\nGarment Type: Socks and Tights\nDescription: Semi shiny nylon stockings with a wide, reinforced trim at the top. Use with a suspender belt. 20 denier.\n","[-0.004496290348470211, -0.00015243499365169555, -0.0019515148596838117, -0.024285517632961273, 0.0063974992372095585, -0.011337867937982082, -0.014834982343018055, -0.013794174417853355, 0.0032542594708502293, -0.054038748145103455, -0.020941054448485374, -0.00012749896268360317, -0.007320349104702473, -0.0029142622370272875, -0.02081615850329399, 0.0037607860285788774, 0.012420307844877243, -0.0021683499217033386, -0.0007493816665373743, -0.0499032698571682, -0.012559082359075546, 0.023078..."
8,111586,\n##Product\nName: Shape Up 30 den 1p Tights\nType: Leggings/Tights\nGroup: Garment Lower body\nGarment Type: Socks and Tights\nDescription: Tights with built-in support to lift the bottom. Black in 30 denier and light amber in 15 denier.\n,"[-0.004459634888917208, -0.0044491165317595005, 0.0038390723057091236, -0.025565767660737038, -0.0011376977199688554, -0.005630639381706715, 0.0020141981076449156, -0.02771143987774849, 0.005038124974817038, -0.04779382050037384, 0.0006499251467175782, 0.008793053217232227, -0.009872902184724808, 0.0015242344234138727, -0.030824771150946617, 0.0005031112814322114, 0.016169682145118713, -0.008274164982140064, 0.00863878894597292, -0.025145046412944794, 0.009964058175683022, 0.0205591972917318..."
27,123173,\n##Product\nName: Control Top 50 den 1p Tights\nType: Leggings/Tights\nGroup: Garment Lower body\nGarment Type: Socks and Tights\nDescription: 50 denier tights with reinforcement at the top for a shaping effect on the tummy and thighs.\n,"[-0.009776857681572437, -0.0016846682410687208, -0.026141706854104996, -0.024725990369915962, -0.00683679198846221, -0.000734578468836844, 0.00196938868612051, -0.015054258517920971, -0.003339550457894802, -0.0461159311234951, 0.0026684864424169064, 0.008031741715967655, -0.005578766576945782, 0.0010539032518863678, -0.028216222301125526, 0.012685385532677174, 0.023043949156999588, -0.01740911416709423, -0.01300076860934496, -0.012089663185179234, 0.00541406637057662, 0.020072344690561295, -..."
35,129085,\n##Product\nName: Pirate Leggings (1)\nType: Leggings/Tights\nGroup: Garment Lower body\nGarment Type: Jersey Basic\nDescription: 3/4-length leggings in stretch jersey with an elasticated waist.\n,"[-0.014741295017302036, -0.006953956559300423, -0.017883554100990295, -0.020206093788146973, -0.0014037701766937971, 0.011346288025379181, -0.00869586132466793, -0.020725248381495476, 0.007596070412546396, -0.053309112787246704, 0.0006856614490970969, -0.0010203804122284055, 0.009433608502149582, -0.008607057854533195, -0.01985088177025318, 0.010731498710811138, 0.016531016677618027, -0.019236091524362564, 0.012241149321198463, -0.005379411391913891, 0.022788209840655327, 0.00856607221066951..."
58,153115,"\n##Product\nName: OP Strapless^\nType: Bra\nGroup: Underwear\nGarment Type: Under-, Nightwear\nDescription: Strapless bra in microfibre with underwired, padded cups that lift and shape the bust. Silicone trim at the top and a hook-and-eye fastening at the back. Detachable, adjustable shoulder straps and side support.\n","[-0.02196011133491993, 0.01499714981764555, -0.021259695291519165, -0.043590616434812546, 0.019378185272216797, 0.00830886047333479, -0.0032050914596766233, -0.0035158153623342514, 0.014557672664523125, -0.030186571180820465, 0.0001450617128284648, 0.013713053427636623, -0.043151140213012695, -0.0118178091943264, -0.019323250278830528, 0.010423842817544937, -0.0014712176052853465, 0.01696106232702732, 0.007395572494715452, -0.01387098990380764, -0.016672655940055847, 0.019529255107045174, 0...."
...,...,...,...
105528,949198,"\n##Product\nName: Saturday jogger\nType: Trousers\nGroup: Garment Lower body\nGarment Type: Jersey Basic\nDescription: Joggers in swe

3. vector properties & index 생성

In [23]:
# load vector properties
# 일단 embedding vector 데이터를 Record로 변환 및 로드
records = product_emb_df[['product_code', 'textEmbedding']].to_dict('records')
print(f'======  loading Product text embeddings ======')
total = len(records)
print(f'staging {total:,} records')

# embedding vector를 Neo4j의 'Product'노드에 저장 (Product에 대한 embedding 결과물이라)
cumulative_count = 0
for recs in gds_db_load.chunks(records, n=100):
    res = gds.run_cypher('''
    UNWIND $recs AS rec
    MATCH(n:Product {product_code: rec.product_code})
    CALL db.create.setNodeVectorProperty(n, "textEmbedding", rec.textEmbedding)
    RETURN count(n) AS propertySetCount
    ''', params={'recs': recs})
    cumulative_count += res.iloc[0, 0]
    print(f'Set {cumulative_count:,} of {total:,} text embeddings')

#코사인 유사도 사용
#create index(필요할때 바로 꺼내올수 있도록)
#Product 노드의 textEmbedding 속성에 대한 vector index를 생성
gds.run_cypher('''
CREATE VECTOR INDEX product_text_embeddings IF NOT EXISTS FOR (n:Product) ON (n.textEmbedding)
OPTIONS {indexConfig: {
 `vector.dimensions`: toInteger($dim),
 `vector.similarity_function`: 'cosine'
}}''', params={'dim': embedding_dimension})

# db.awaitIndex procedure description : Wait for an index to come online (for example: CALL db.awaitIndex("MyIndex", 300)).
'''
When you create an index in Neo4j, it might take some time for the index to become fully populated, especially if it's being created on a large dataset.
The CALL db.awaitIndex procedure allows you to specify an index by its name and a timeout period.
The operation will wait for the specified index to come online up to the timeout limit.
If the index is available before the timeout, the procedure returns immediately;
if the timeout is reached and the index is still not online, an error will occur.
'''
gds.run_cypher('CALL db.awaitIndex("product_text_embeddings", 300)')

======  loading Product text embeddings ======
staging 4,245 records
Set 100 of 4,245 text embeddings
Set 200 of 4,245 text embeddings
Set 300 of 4,245 text embeddings
Set 400 of 4,245 text embeddings
Set 500 of 4,245 text embeddings
Set 600 of 4,245 text embeddings
Set 700 of 4,245 text embeddings
Set 800 of 4,245 text embeddings
Set 900 of 4,245 text embeddings
Set 1,000 of 4,245 text embeddings
Set 1,100 of 4,245 text embeddings
Set 1,200 of 4,245 text embeddings
Set 1,300 of 4,245 text embeddings
Set 1,400 of 4,245 text embeddings
Set 1,500 of 4,245 text embeddings
Set 1,600 of 4,245 text embeddings
Set 1,700 of 4,245 text embeddings
Set 1,800 of 4,245 text embeddings
Set 1,900 of 4,245 text embeddings
Set 2,000 of 4,245 text embeddings
Set 2,100 of 4,245 text embeddings
Set 2,200 of 4,245 text embeddings
Set 2,300 of 4,245 text embeddings
Set 2,400 of 4,245 text embeddings
Set 2,500 of 4,245 text embeddings
Set 2,600 of 4,245 text embeddings
Set 2,700 of 4,245 text embeddings
Set 

""


In [24]:
gds.run_cypher('''
MATCH (n:Product)
SET n.text = trim(
  coalesce(n.product_name,'') + ' ' +
  coalesce(n.product_type_name,'') + ' ' +
  coalesce(n.product_group_name,'') + ' ' +
  coalesce(n.garment_group_name,'') + ' ' +
  coalesce(n.detail_desc,'')
)
RETURN count(n) AS updated
''')


,updated
0,2771


4. vector search → prompt를 이용하여 검색(use the vector index to find semantic similar products in user searches) 

In [40]:
# 1) 검색어 정의
# 사용자가 입력하는 검색 쿼리 (텍스트 형태)
# 다른 검색어 예: 'denim jeans, loose fit, high-waist'
search_prompt = 'loose fit'

# 2) 검색어를 임베딩 벡터로 변환
# OpenAI 임베딩 모델을 사용하여 검색어를 1536차원 벡터로 변환
# -> 이렇게 해야 제품 설명 임베딩 벡터와 직접 비교 가능
query_vector = embedding_model.embed_query(search_prompt)

# 임베딩 벡터의 길이와 앞 10개 값 출력 (디버깅/확인용)
print(f'query vector length: {len(query_vector)}')   # 보통 1536 나옴
print(f'query vector sample: {query_vector[:10]}')   # 일부 벡터 값 확인

query vector length: 1536
query vector sample: [-0.03052879311144352, -0.015792623162269592, -0.0066840993240475655, -0.013821931555867195, -0.01682198792695999, 0.011824151501059532, -0.0021789351012557745, -0.020113246515393257, -0.005360146518796682, -0.01611768640577793]


In [41]:
#쿼리 벡터와 제품 텍스트 임베딩 벡터간의 유사도 계산 -> score
gds.run_cypher('''
CALL db.index.vector.queryNodes("product_text_embeddings", 10, $queryVector)
YIELD node AS product, score
RETURN product.product_code AS product_code,
    product.text AS text,
    score
''', params={'queryVector': query_vector})

,product_code,text,score
0,864216,"Trousers Garment Lower body Trousers Denim 5-pocket, ankle-length jeans in washed denim with an extra-high waist and zip fly and button. Loose fit with gently tapered legs. The cotton content of the jeans is partly recycled.",0.908569
1,917606,"Trousers Garment Lower body Trousers Denim 5-pocket, ankle-length jeans in washed cotton denim with an extra-high waist and zip fly and button. Slightly looser fit with straight legs. The cotton content of the jeans is partly recycled.",0.908463
2,868213,Vest top Garment Upper body Jersey Fancy Loose-fitting sports vest top in fast-drying functional fabric with a racer back and rounded hem. The polyester content of the top is recycled.,0.908432
3,804992,T-shirt Garment Upper body Jersey Fancy Wide sports top in fast-drying functional fabric with a slightly wider neckline and short cap sleeves. Rounded and slightly longer at the back.,0.908340
4,851094,Vest top Garment Upper body Jersey Fancy Sports vest top in ribbed fast-drying functional fabric made from recycled polyester with short slits in the sides. Longer at the back.,0.907349
5,864224,"Trousers Garment Lower body Trousers Denim 5-pocket jeans in washed denim with a high waist, zip fly and button and straight, wide legs. The cotton content of the jeans is partly recycled.",0.905380
6,766402,"Trousers Garment Lower body Trousers Ankle-length joggers in woven fabric with an elasticated, drawstring waist, diagonal side pockets and welt back pockets. Legs with creases and sewn-in turn-ups at the hems. Slim Fit – a fit that is close-fitting at the thighs, knees and ankles to create a fitted silhouette.",0.904953
7,814620,"Trousers Garment Lower body Trousers Denim 5-pocket denim jeans with a regular waist, zip fly and skinny legs. Made using Lycra® Freefit® technology for soft, super-generous stretch, maximum mobility and optimal comfort.",0.904297
8,572998,"Trousers Garment Lower body Trousers 5-pocket, ankle-length jeans in washed denim with a high waist, zip fly and button and gently tapered legs.",0.904007
9,912204,Vest top Garment Upper body Jersey Fancy Loose-fitting sports vest top in fast-drying jersey with a racer back.,0.902283


Langchain

In [42]:
from langchain.vectorstores.neo4j_vector import Neo4jVector

kg_vector_search = Neo4jVector.from_existing_index(
    embedding=embedding_model,
    url="bolt://127.0.0.1:7687",
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name='product_text_embeddings')

In [43]:
res = kg_vector_search.similarity_search(search_prompt, k=10)
res

[Document(metadata={'product_type_name': 'Trousers', 'detail_desc': '5-pocket, ankle-length jeans in washed denim with an extra-high waist and zip fly and button. Loose fit with gently tapered legs. The cotton content of the jeans is partly recycled.', 'product_code': 864216, 'garment_group_name': 'Trousers Denim', 'product_group_name': 'Garment Lower body', 'prod_name': 'Loose Mom Ultra HW Consc.'}, page_content='Trousers Garment Lower body Trousers Denim 5-pocket, ankle-length jeans in washed denim with an extra-high waist and zip fly and button. Loose fit with gently tapered legs. The cotton content of the jeans is partly recycled.'),
 Document(metadata={'product_type_name': 'T-shirt', 'detail_desc': 'Wide sports top in fast-drying functional fabric with a slightly wider neckline and short cap sleeves. Rounded and slightly longer at the back.', 'product_code': 804992, 'garment_group_name': 'Jersey Fancy', 'product_group_name': 'Garment Upper body', 'prod_name': 'Izzy loose tee'}, pa

In [44]:
#가독성을 위해 데이터 프레임으로 바꿔주기
# Visualize as a dataframe
pd.DataFrame([{'document': d.page_content} for d in res])

,document
0,"Trousers Garment Lower body Trousers Denim 5-pocket, ankle-length jeans in washed denim with an extra-high waist and zip fly and button. Loose fit with gently tapered legs. The cotton content of the jeans is partly recycled."
1,T-shirt Garment Upper body Jersey Fancy Wide sports top in fast-drying functional fabric with a slightly wider neckline and short cap sleeves. Rounded and slightly longer at the back.
2,"Trousers Garment Lower body Trousers Denim 5-pocket, ankle-length jeans in washed cotton denim with an extra-high waist and zip fly and button. Slightly looser fit with straight legs. The cotton content of the jeans is partly recycled."
3,Vest top Garment Upper body Jersey Fancy Loose-fitting sports vest top in fast-drying functional fabric with a racer back and rounded hem. The polyester content of the top is recycled.
4,Vest top Garment Upper body Jersey Fancy Sports vest top in ribbed fast-drying functional fabric made from recycled polyester with short slits in the sides. Longer at the back.
5,"Trousers Garment Lower body Trousers Denim 5-pocket jeans in washed denim with a high waist, zip fly and button and straight, wide legs. The cotton content of the jeans is partly recycled."
6,"Trousers Garment Lower body Trousers Ankle-length joggers in woven fabric with an elasticated, drawstring waist, diagonal side pockets and welt back pockets. Legs with creases and sewn-in turn-ups at the hems. Slim Fit – a fit that is close-fitting at the thighs, knees and ankles to create a fitted silhouette."
7,"Trousers Garment Lower body Trousers Denim 5-pocket denim jeans with a regular waist, zip fly and skinny legs. Made using Lycra® Freefit® technology for soft, super-generous stretch, maximum mobility and optimal comfort."
8,"Trousers Garment Lower body Trousers 5-pocket, ankle-length jeans in washed denim with a high waist, zip fly and button and gently tapered legs."
9,Vest top Garment Upper body Jersey Fancy Loose-fitting sports vest top in fast-drying jersey with a racer back.


5. semantic search with context (Graph Patterns) - 패턴매칭

In [101]:
custid_list[0]

'0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf4672f30b3e622fec55'

In [107]:
CUSTOMER_ID = "0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf4672f30b3e622fec55"

# 특정 고객의 구매 이력을 반영
# search performance is highly relies on magic_query and objective
magic_query = f"""
    WITH node AS product, score AS searchScore

    OPTIONAL MATCH(product)<-[:VARIANT_OF]-(:Article)<-[:PURCHASED]-(:Customer)
    -[:PURCHASED]->(a:Article)<-[:PURCHASED]-(:Customer {{customerId: '{CUSTOMER_ID}'}})

    WITH count(a) AS purchaseScore, product.text AS text, searchScore, product.product_code AS product_code
    RETURN text,
        (1+purchaseScore)*searchScore AS score,
        {{product_code: product_code, purchaseScore:purchaseScore, searchScore:searchScore}} AS metadata
    ORDER BY purchaseScore DESC, searchScore DESC LIMIT 15
    """
    # 다른 고객들이 동일한 Article을 얼마나 구매했는지를 세어 purchaseScore로 저장
    #(1+purchaseScore)*searchScore AS score: 구매 횟수 -> 구매 횟수가 많은 상품일수록 점수가 높아짐

#기존의 벡터 인덱스를 사용하여 Neo4jVector 객체를 생성하고, 개인화된 검색을 수행하기 위해 앞서 정의한 magic_query를 사용
kg_personalized_search = Neo4jVector.from_existing_index(
    embedding=embedding_model,
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    index_name='product_text_embeddings',
    retrieval_query=magic_query)

In [47]:
res = kg_personalized_search.similarity_search(search_prompt, k=100)

# Visualize as a dataframe
pd.DataFrame([{'product_code': d.metadata['product_code'],
               'document': d.page_content,
               'searchScore': d.metadata['searchScore'],} for d in res])

,product_code,document,searchScore
0,905518,"T-shirt Garment Upper body Jersey Fancy Wide, straight-cut sports top in fast-drying functional fabric. Loose fit with a round, slightly wider neckline, short sleeves, and slits in the sides. Slightly longer at the back.",0.910660
1,864216,"Trousers Garment Lower body Trousers Denim 5-pocket, ankle-length jeans in washed denim with an extra-high waist and zip fly and button. Loose fit with gently tapered legs. The cotton content of the jeans is partly recycled.",0.908447
2,804992,T-shirt Garment Upper body Jersey Fancy Wide sports top in fast-drying functional fabric with a slightly wider neckline and short cap sleeves. Rounded and slightly longer at the back.,0.907959
3,917606,"Trousers Garment Lower body Trousers Denim 5-pocket, ankle-length jeans in washed cotton denim with an extra-high waist and zip fly and button. Slightly looser fit with straight legs. The cotton content of the jeans is partly recycled.",0.907715
4,868213,Vest top Garment Upper body Jersey Fancy Loose-fitting sports vest top in fast-drying functional fabric with a racer back and rounded hem. The polyester content of the top is recycled.,0.907547
...,...,...,...
10,912204,Vest top Garment Upper body Jersey Fancy Loose-fitting sports vest top in fast-drying jersey with a racer back.,0.901779
11,573152,T-shirt Garment Upper body Jersey Fancy Sports top in fast-drying functional fabric with short cap sleeves.,0.900757
12,891638,"Trousers Garment Lower body Trousers Denim 5-pocket jeans in washed, stretch denim with a regular waist, zip fly and button and gently tapered legs with good room for movement over the thighs and knees.",0.900711
13,926825,"Trousers Garment Lower body Trousers Tailored trousers in a soft weave. High waist with elastication at the back, a button at the front and fly. Front pockets, a fake back pocket and straight legs with pleats for added width.",0.900513


6. Augmenting Semantic Search with Knowledge Graph Inference & ML

In [85]:
%time
from neo4j_tools import gds_utils

# 1. 과거 GDS 분석 제거
gds_utils.clear_all_gds_graphs(gds)
gds_utils.delete_relationships('CUSTOMERS_ALSO_LIKE', gds, src_node_label='Article')

# 2. GDS용 그래프 생성 (노드, 관계 명세 명확히 전달)
G, result = gds.graph.project(
    graph_name='proj',
    node_spec={
        'Article': {}
    },
    relationship_spec={
        'PURCHASED': {
            'orientation': 'UNDIRECTED'  # 또는 'NATURAL' → 실험해보세요
        }
    }
)

# 3. FastRP 임베딩 생성
gds.fastRP.mutate(
    G,
    mutateProperty='embedding',
    embeddingDimension=128,
    randomSeed=7474,
    concurrency=4,
    iterationWeights=[0.0, 1.0, 1.0]
)

# 4. KNN으로 CUSTOMERS_ALSO_LIKE 생성
knn_stats = gds.knn.write(
    G,
    nodeProperties=['embedding'],
    nodeLabels=['Article'],
    writeRelationshipType='CUSTOMERS_ALSO_LIKE',
    writeProperty='score',
    sampleRate=1.0,
    initialSampler='randomWalk',
    concurrency=1,
    similarityCutoff=0.75,
    randomSeed=7474
)

# 5. 임베딩을 DB에 저장
gds.graph.writeNodeProperties(G, ['embedding'], ['Article'])

# 6. 그래프 정리
gds.graph.drop(G)

# 7. 결과 출력
knn_stats

CPU times: total: 0 ns
Wall time: 5.72 μs


preProcessingMillis                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         0
computeMillis                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [86]:
from langchain.graphs import Neo4jGraph
import pandas as pd

kg = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

CUSTOMER_ID = "여기에_고객_ID_입력"  # 예: "12345"

# 개선된 쿼리
query = '''
MATCH (c:Customer {customerId: $customerId})-[:PURCHASED]->(a1:Article)
MATCH (a1)-[r:CUSTOMERS_ALSO_LIKE]->(a2:Article)
MATCH (a2)-[:VARIANT_OF]->(p:Product)
RETURN 
  p.product_code AS product_code,
  p.prodName AS prodName,
  p.productTypeName AS productType,
  p.text AS document,
  sum(r.score) AS recommenderScore
ORDER BY recommenderScore DESC
LIMIT $k
'''

res = kg.query(query, params={'customerId': CUSTOMER_ID, 'k': 15})

# 결과 출력
pd.DataFrame(res)

""


7. LLM For Generating Grounded Content

In [87]:
# Import relevant libraries
from langchain.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain.schema import StrOutputParser

llm = ChatOpenAI(
    model="gpt-5",   # 사용 모델은 그대로/또는 mini 권장
    temperature=0,
    streaming=False        # ★ 반드시 False
)


In [88]:
# This will be a function so we can change per customer id
# We will use a mock URL for our sources in the metadata
# 특정 고객 ID에 따라 개인화된 검색을 생성
def kg_personalized_search_gen(customer_id):
    return Neo4jVector.from_existing_index(
        embedding=embedding_model,
        url=NEO4J_URI,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD,
        index_name='product_text_embeddings',
        retrieval_query=f"""
        WITH node AS product, score AS searchScore

        OPTIONAL MATCH(product)<-[:VARIANT_OF]-(:Article)<-[:PURCHASED]-(:Customer)
        -[:PURCHASED]->(a:Article)<-[:PURCHASED]-(:Customer {{customerId: '{customer_id}'}})
        WITH count(a) AS purchaseScore, product, searchScore
        RETURN product.text + '\nurl: ' + 'https://representative-domain/product/' + product.product_code  AS text,
            (1.0+purchaseScore)*searchScore AS score,
            {{source: 'https://representative-domain/product/' + product.product_code}} AS metadata
        ORDER BY purchaseScore DESC, searchScore DESC LIMIT 5

    """
    )

# Use the same personalized recommendations as above but with a smaller limit
# 해당 고객에 대한 추천 결과를 생성
def kg_recommendations_app(customer_id, k=30):
    res = kg.query("""
    MATCH(:Customer {customerId:$customerId})-[:PURCHASED]->(:Article)
    -[r:CUSTOMERS_ALSO_LIKE]->(:Article)-[:VARIANT_OF]->(product)
    RETURN product.text + '\nurl: ' + 'https://representative-domain/product/' + product.product_code  AS text,
        sum(r.score) AS recommenderScore
    ORDER BY recommenderScore DESC LIMIT $k
    """, params={'customerId': customer_id, 'k':k})

    return "\n\n".join([d['text'] for d in res])
    #쿼리는 고객이 구매한 상품과 유사한 상품을 CUSTOMERS_ALSO_LIKE 관계를 통해 찾고, 이 관계의 score 값을 합산하여 최종 점수를 계산

In [52]:
general_system_template = '''
You are a personal assistant named Sally for a fashion, home, and beauty company called HM.
write an email to {customerName}, one of your customers, to promote and summarize products relevant for them given the current season / time of year: {timeOfYear} .
Please only mention the products listed below. Do not come up with or add any new products to the list.
Each product comes with an https `url` field. Make sure to provide that https url with descriptive name text in markdown for each product.

---
# Relevant Products:
{searchProds}

# Customer May Also Be Interested In the following
 (pick items from here that pair with the above products well for the current season / time of year: {timeOfYear}.
 prioritize those higher in the list if possible):
{recProds}
---
'''
general_user_template = "{searchPrompt}"
messages = [
    SystemMessagePromptTemplate.from_template(general_system_template),
    HumanMessagePromptTemplate.from_template(general_user_template),
]

#위의 두 가지 프롬프트를 결합하여 최종 프롬프트를 생성
prompt = ChatPromptTemplate.from_messages(messages)

In [92]:
from langchain.schema.runnable import RunnableLambda

# helper function
def format_docs(docs):
    return "\n\n".join([d.page_content for d in docs])

# LLM chain
def chain_gen(customer_id):
    return ({'searchProds': (lambda x:x['searchPrompt']) | kg_personalized_search_gen(customer_id).as_retriever(search_kwargs={"k": 100}) | format_docs,
             'recProds': (lambda x:customer_id) |  RunnableLambda(kg_recommendations_app),
             'customerName': lambda x:x['customerName'],
             'timeOfYear': lambda x:x['timeOfYear'],
             "searchPrompt":  lambda x:x['searchPrompt']}
            | prompt
            | llm
            | StrOutputParser())

In [54]:
#예시
chain = chain_gen(CUSTOMER_ID)
print(chain.invoke({'searchPrompt':search_prompt, 'customerName':'Alex Smith', 'timeOfYear':'Feb, 2024'}))

Subject: Alex, here are easy loose-fit layers for February

Hi Alex,

With February’s cool, in‑between weather, here’s a curated loose‑fit lineup to keep you comfy for layering, lounging, and getting those workouts in.

- [Loose-Fit Fast-Drying Sports T-Shirt (wide neckline, side slits)](https://representative-domain/product/905518) – Straight-cut, slightly longer in back for easy layering.
- [Loose-Fit Fast-Drying Sports Tee (cap sleeves, rounded hem)](https://representative-domain/product/804992) – A breathable go-to for studio-to-street.
- [Loose-Fit Racerback Vest Top (fast-drying, recycled polyester)](https://representative-domain/product/868213) – Lightweight base layer under hoodies or on its own for training.
- [Extra-High Waist Loose-Fit Ankle Jeans (tapered leg, partly recycled cotton)](https://representative-domain/product/864216) – Relaxed shape that works with chunky sneakers or ankle boots.
- [Extra-High Waist Slightly Looser Straight-Leg Ankle Jeans (partly recycled cott

In [124]:
custid = custid_list[3]

In [125]:
# LLM chain 실행
chain = chain_gen(custid)

result = chain.invoke({
    'searchPrompt': search_prompt,
    'customerName': 'Alex Smith',
    'timeOfYear': 'Feb, 2024'
})

# 결과에서 product_code 추출 (정규식 이용)
import re

# 숫자로만 된 product_code 찾기
product_codes = re.findall(r'/product/(\d+)', result)
print("추천된 product_code 리스트:", product_codes)


추천된 product_code 리스트: ['905518', '804992', '864216', '917606', '868213']


In [129]:
# tr_test_merged의 product_code는 숫자일 수 있으니 문자열로 변환
matched_rows = tr_test_merged[
    tr_test_merged['product_code'].astype(str).isin(product_codes)
]

# 일치하는 article_id만 추출
matched_article_ids = matched_rows['article_id'].unique().tolist()

print("추천된 product_codes와 일치하는 article_id 목록:")
print(matched_article_ids)

추천된 product_codes와 일치하는 article_id 목록:
[804992013, 864216017, 804992014, 905518001, 804992017, 917606004, 804992016, 804992033, 804992034, 864216001, 864216005, 868213002, 905518002, 917606001, 905518003, 868213011, 868213017, 804992001, 917606003]


In [133]:
custid

'001436e2c83cda28548dd668cfc7d621d70d2baf6f6cf0684c7cc8499d0c351f'

In [121]:
ar = pd.read_csv('Desktop/H&M/articles.csv')
# tr_test에 product_code 붙이기 (article_id 기준으로 조인)
tr_test_merged = tr_test.merge(
    ar[['article_id', 'product_code']],  # 필요한 컬럼만
    on='article_id',
    how='left'
)

# 확인
tr_test_merged.head()

,t_dat,customer_id,article_id,price,sales_channel_id,product_code
0,2020-09-22,0003e867a930d0d6842f923d6ba7c9b77aba33fe2a0fbf4672f30b3e622fec55,827487003,0.042356,2,827487
1,2020-09-22,000525e3fe01600d717da8423643a8303390a055c578ed8a97256600baf54565,874110016,0.025407,2,874110
2,2020-09-22,0010e8eb18f131e724d6997909af0808adbba057529edb1523944f7d4e02b4ce,610776002,0.008458,1,610776
3,2020-09-22,0010e8eb18f131e724d6997909af0808adbba057529edb1523944f7d4e02b4ce,372860001,0.013542,1,372860
4,2020-09-22,001436e2c83cda28548dd668cfc7d621d70d2baf6f6cf0684c7cc8499d0c351f,687524001,0.025407,2,687524


In [127]:
tr_test_merged[tr_test_merged['customer_id'] == custid]

,t_dat,customer_id,article_id,price,sales_channel_id,product_code
4,2020-09-22,001436e2c83cda28548dd668cfc7d621d70d2baf6f6cf0684c7cc8499d0c351f,687524001,0.025407,2,687524
5,2020-09-22,001436e2c83cda28548dd668cfc7d621d70d2baf6f6cf0684c7cc8499d0c351f,871517012,0.025407,2,871517


In [143]:
import numpy as np
import re

# ===== 평가 지표 함수들 =====
def hit_at_k(pred, true, k=10):
    return int(len(set(pred[:k]) & set(true)) > 0)

def recall_at_k(pred, true, k=10):
    return len(set(pred[:k]) & set(true)) / len(set(true)) if true else 0

def precision_at_k(pred, true, k=10):
    return len(set(pred[:k]) & set(true)) / k

def average_precision(pred, true, k=10):
    score, hits = 0.0, 0
    for i, p in enumerate(pred[:k], start=1):
        if p in true:
            hits += 1
            score += hits / i
    return score / min(len(true), k) if true else 0

def mean_reciprocal_rank(pred, true, k=10):
    for i, p in enumerate(pred[:k], start=1):
        if p in true:
            return 1 / i
    return 0

def ndcg_at_k(pred, true, k=10):
    dcg = 0.0
    for i, p in enumerate(pred[:k], start=1):
        if p in true:
            dcg += 1 / np.log2(i + 1)
    idcg = sum(1 / np.log2(i + 1) for i in range(1, min(len(true), k) + 1))
    return dcg / idcg if idcg > 0 else 0


# ===== 전체 고객 평가=====
def evaluate_all_customers_offline(custid_list, tr_test_merged, k=10, sample_size=100):
    metrics = {"Hit": [], "Recall": [], "Precision": [], "MAP": [], "MRR": [], "NDCG": []}

    # 랜덤으로 고객 샘플링
    sampled_customers = np.random.choice(custid_list, size=min(sample_size, len(custid_list)), replace=False)

    for custid in sampled_customers:
        # 1. 실제 구매 product_code
        true_codes = tr_test_merged.loc[
            tr_test_merged['customer_id'] == custid, 'product_code'
        ].astype(str).tolist()

        if not true_codes:
            continue

        # 2. 추천 product_codes → 지금은 LLM chain 안 쓰고, 예시로 product_codes 리스트 사용
        #    (실제로는 고객별 추천 결과 dict 만들어서 여기에 불러오면 됨)
        pred_codes = product_codes  # 👈 여기에 이미 추출한 추천 리스트 넣으세요

        if not pred_codes:
            continue

        # 3. 성능평가 지표 계산
        metrics["Hit"].append(hit_at_k(pred_codes, true_codes, k))
        metrics["Recall"].append(recall_at_k(pred_codes, true_codes, k))
        metrics["Precision"].append(precision_at_k(pred_codes, true_codes, k))
        metrics["MAP"].append(average_precision(pred_codes, true_codes, k))
        metrics["MRR"].append(mean_reciprocal_rank(pred_codes, true_codes, k))
        metrics["NDCG"].append(ndcg_at_k(pred_codes, true_codes, k))

    results = {m: np.mean(v) if v else 0.0 for m, v in metrics.items()}
    return results


# 실행 예시
results = evaluate_all_customers_offline(custid_list, tr_test_merged, k=10, sample_size=10000)
print("랜덤 샘플 고객 평균 성능")
for metric, value in results.items():
    print(f"{metric}@10: {value:.4f}")

랜덤 샘플 고객 평균 성능
Hit@10: 0.0139
Recall@10: 0.0056
Precision@10: 0.0014
MAP@10: 0.0028
MRR@10: 0.0073
NDCG@10: 0.0044


Hit@10: 0.0139
→ 고객 100명 중 약 1.4명만 실제 구매 상품이 추천 목록에 포함됨. (추천 적중률이 매우 낮음)

Recall@10: 0.0056
→ 실제 구매한 상품 중 평균적으로 0.56%만 추천에 잡힘.

Precision@10: 0.0014
→ 추천 10개 중 평균적으로 0.14%만 맞는 상품. (거의 대부분 불필요한 추천)

MAP@10: 0.0028
→ 적중이 되더라도 순위 상에서 일관적으로 좋은 품질을 보여주지 못함.

MRR@10: 0.0073
→ 올바른 추천이 리스트 상단에 잘 올라오지 못함.

NDCG@10: 0.0044
→ 추천 순위의 품질(정답이 얼마나 상위에 위치하는지)도 낮음.

In [141]:
def evaluate_all_customers_offline(custid_list, tr_test_merged, k=10):
    metrics = {"Hit": [], "Recall": [], "Precision": [], "MAP": [], "MRR": [], "NDCG": []}

    for custid in custid_list:
        # 1. 실제 구매 product_code
        true_codes = tr_test_merged.loc[
            tr_test_merged['customer_id'] == custid, 'product_code'
        ].astype(str).tolist()

        if not true_codes:
            continue

        # 2. 추천 product_codes → 지금은 LLM chain 안 쓰고 product_codes 사용
        pred_codes = product_codes  # 👈 고객별 추천 결과 딕셔너리 있으면 교체하세요

        if not pred_codes:
            continue

        # 3. 성능평가 지표 계산
        metrics["Hit"].append(hit_at_k(pred_codes, true_codes, k))
        metrics["Recall"].append(recall_at_k(pred_codes, true_codes, k))
        metrics["Precision"].append(precision_at_k(pred_codes, true_codes, k))
        metrics["MAP"].append(average_precision(pred_codes, true_codes, k))
        metrics["MRR"].append(mean_reciprocal_rank(pred_codes, true_codes, k))
        metrics["NDCG"].append(ndcg_at_k(pred_codes, true_codes, k))

    results = {m: np.mean(v) if v else 0.0 for m, v in metrics.items()}
    return results


In [142]:
results = evaluate_all_customers_offline(custid_list, tr_test_merged, k=10)
print("전체 고객 평균 성능")
for metric, value in results.items():
    print(f"{metric}@10: {value:.4f}")

전체 고객 평균 성능
Hit@10: 0.0134
Recall@10: 0.0054
Precision@10: 0.0014
MAP@10: 0.0026
MRR@10: 0.0070
NDCG@10: 0.0042
